<a href="https://colab.research.google.com/github/yhshengjy/ClinPKPD/blob/main/Notebook5_vancomycin_PKPD_simulation_english.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 5: Vancomycin PK/PD Simulation

This notebook is the first module in the "Antibacterial PK/PD Applications" section of the interactive clinical pharmacy PK/PD simulation platform.

Using **vancomycin** as the example, this section explains how to connect pharmacokinetic parameters, dosing regimens, pathogen MIC, and PK/PD indices.

Vancomycin is commonly used for serious Gram-positive bacterial infections, especially MRSA-associated infections. For serious MRSA infections, the commonly used clinical PK/PD index is:

$$
AUC_{24}/MIC
$$

The 2020 ASHP/IDSA/PIDS/SIDP consensus guideline recommends that, for suspected or confirmed serious MRSA infections, the daily vancomycin AUC should generally be maintained within **400--600 mg·h/L** to balance efficacy and nephrotoxicity risk. This recommendation usually assumes an MIC of 1 mg/L.

The core logic of this notebook is:

$$
Dose\ regimen \rightarrow Concentration(t) \rightarrow AUC_{24} \rightarrow AUC_{24}/MIC \rightarrow Target\ attainment
$$

Note: This notebook is intended for educational simulation only and should not be used for real patient prescribing decisions. Real clinical dosing requires consideration of infection site, microbiology, MIC testing method, renal function, body weight, therapeutic drug monitoring, concomitant medications, and local institutional protocols.

## 1. Learning Objectives

After completing this notebook, you should be able to:

1. Explain the vancomycin PK/PD index AUC24/MIC and its relationship to antibacterial efficacy.
2. Describe why trough concentration alone may not fully represent vancomycin exposure.
3. Use a one-compartment intermittent intravenous infusion model to simulate vancomycin concentration-time profiles.
4. Calculate and interpret Cmax, Cmin, AUC24, and AUC24/MIC under different dosing regimens.
5. Evaluate how dose, dosing interval, clearance, volume of distribution, renal function, and MIC influence efficacy and safety.

## 2. Core Concepts in Vancomycin PK/PD

The clinical PK/PD evaluation of vancomycin usually focuses not on a single peak concentration, but on the ratio of 24-hour drug exposure to MIC:

$$
AUC_{24}/MIC
$$

where:

| Symbol | Meaning | Common unit |
|---|---|---|
| AUC24 | Area under the concentration-time curve over 24 hours | mg·h/L |
| MIC | Minimum inhibitory concentration | mg/L |
| AUC24/MIC | Ratio of exposure to pathogen susceptibility | Dimensionless; often written as mg·h/L divided by mg/L |

It can be understood as follows:

- AUC24 represents the patient's total drug exposure over 24 hours.
- MIC represents the pathogen's susceptibility to the drug.
- The higher the MIC, the lower the AUC24/MIC for the same AUC24.
- When MIC is high, simply increasing the dose may increase AUC24/MIC, but it may also push AUC24 above the safety range.

In this educational simulation, we use the following simplified criteria:

| Assessment item | Educational threshold |
|---|---|
| Efficacy-related target | AUC24/MIC ≥ 400 |
| Upper exposure limit for safety | AUC24 ≤ 600 mg·h/L |
| Ideal educational range | When MIC = 1 mg/L, AUC24 is approximately 400--600 mg·h/L |

Note: These thresholds come from vancomycin therapeutic drug monitoring consensus recommendations for serious MRSA infections. They should not be mechanically applied to all infection types, all patient populations, or all clinical scenarios.

## 3. One-Compartment Intermittent Intravenous Infusion Model

Vancomycin is usually administered by intermittent intravenous infusion rather than rapid intravenous bolus injection.

In a one-compartment model, let:

$$
k = \frac{CL}{V_d}
$$

The infusion rate is:

$$
R_0 = \frac{Dose}{T_{inf}}
$$

where:

| Symbol | Meaning | Common unit |
|---|---|---|
| Dose | Dose per administration | mg |
| Tinf | Infusion duration | h |
| tau | Dosing interval | h |
| Vd | Apparent volume of distribution | L |
| CL | Clearance | L/h |
| k | Elimination rate constant | 1/h |
| R0 | Infusion rate | mg/h |

For a single infusion, if the time after dosing is $t$:

### During infusion: $0 \leq t \leq T_{inf}$

$$
C(t) = \frac{R_0}{CL}\left(1-e^{-kt}\right)
$$

### After the end of infusion: $t > T_{inf}$

$$
C(t) = \frac{R_0}{CL}\left(1-e^{-kT_{inf}}\right)e^{-k(t-T_{inf})}
$$

During multiple dosing, the concentration-time profiles generated by each dose can be added together. This is called the **superposition principle**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, fixed

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True


def infusion_contribution(t_after_dose, dose_mg, infusion_h, vd_l, cl_l_h):
    """
    Concentration contribution from one IV infusion dose in a one-compartment model.
    """
    k_elim = cl_l_h / vd_l
    infusion_rate = dose_mg / infusion_h
    concentration = np.zeros_like(t_after_dose, dtype=float)

    during = (t_after_dose >= 0) & (t_after_dose <= infusion_h)
    after = t_after_dose > infusion_h

    concentration[during] = (
        infusion_rate / cl_l_h
        * (1 - np.exp(-k_elim * t_after_dose[during]))
    )

    concentration[after] = (
        infusion_rate / cl_l_h
        * (1 - np.exp(-k_elim * infusion_h))
        * np.exp(-k_elim * (t_after_dose[after] - infusion_h))
    )

    return concentration


def simulate_multiple_iv_infusions(
    dose_mg,
    tau_h,
    infusion_h,
    vd_l,
    cl_l_h,
    n_doses,
    dt=0.01
):
    """
    Simulate multiple intermittent IV infusions using superposition.
    """
    total_time_h = tau_h * n_doses
    t = np.arange(0, total_time_h + dt, dt)
    concentration = np.zeros_like(t, dtype=float)

    dose_times = np.arange(0, n_doses * tau_h, tau_h)

    for dose_time in dose_times:
        concentration += infusion_contribution(
            t_after_dose=t - dose_time,
            dose_mg=dose_mg,
            infusion_h=infusion_h,
            vd_l=vd_l,
            cl_l_h=cl_l_h
        )

    return t, concentration, dose_times


def calculate_vancomycin_metrics(
    t,
    concentration,
    dose_mg,
    tau_h,
    infusion_h,
    vd_l,
    cl_l_h,
    mic_mg_l,
    n_doses
):
    """
    Calculate PK/PD metrics for vancomycin teaching simulation.
    """
    k_elim = cl_l_h / vd_l
    half_life = np.log(2) / k_elim

    # Steady-state AUC24 for linear PK
    daily_dose_mg = dose_mg * (24 / tau_h)
    auc24_ss = daily_dose_mg / cl_l_h
    auc_mic = auc24_ss / mic_mg_l

    # Last dosing interval metrics
    last_dose_time = (n_doses - 1) * tau_h
    interval_mask = (t >= last_dose_time) & (t <= last_dose_time + tau_h)
    t_interval = t[interval_mask] - last_dose_time
    c_interval = concentration[interval_mask]

    cmax_last = np.max(c_interval)
    tmax_last = t_interval[np.argmax(c_interval)]
    cmin_last = c_interval[-1]
    c_end_infusion = np.interp(infusion_h, t_interval, c_interval)

    if auc_mic < 400:
        target_status = "Below efficacy target"
    elif auc24_ss > 600:
        target_status = "High exposure risk"
    else:
        target_status = "Within teaching target range"

    metrics = {
        "Elimination rate constant": k_elim,
        "Half-life": half_life,
        "Daily dose": daily_dose_mg,
        "AUC24 at steady state": auc24_ss,
        "AUC24/MIC": auc_mic,
        "Cmax in last interval": cmax_last,
        "Tmax in last interval": tmax_last,
        "Concentration at end of infusion": c_end_infusion,
        "Cmin before next dose": cmin_last,
        "Target status": target_status
    }

    return metrics


def make_metrics_table(metrics):
    """
    Format metrics dictionary as a dataframe.
    """
    rows = []
    for key, value in metrics.items():
        if isinstance(value, str):
            formatted = value
        elif "rate constant" in key:
            formatted = f"{value:.4f} 1/h"
        elif "Half-life" in key:
            formatted = f"{value:.2f} h"
        elif "Daily dose" in key:
            formatted = f"{value:.0f} mg/day"
        elif "AUC" in key and "MIC" not in key:
            formatted = f"{value:.1f} mg*h/L"
        elif "AUC24/MIC" in key:
            formatted = f"{value:.1f}"
        else:
            formatted = f"{value:.2f} mg/L"
        rows.append({"Metric": key, "Value": formatted})
    return pd.DataFrame(rows)

## 4. Interactive Simulation 1: Vancomycin Multiple-Dose Concentration Profile and AUC/MIC

The following simulation shows the multiple-dose concentration-time profile after intermittent intravenous infusion of vancomycin.

You can adjust:

- Dose per administration
- Dosing interval tau
- Infusion duration Tinf
- Volume of distribution Vd
- Clearance CL
- MIC
- Number of doses

Focus on the following observations:

- After repeated dosing, does the concentration gradually approach steady state?
- When the dose increases, how do AUC24 and AUC24/MIC change?
- When CL decreases, how do AUC24 and trough concentration change?
- When MIC increases, does AUC24/MIC decrease even if AUC24 remains unchanged?

In [ ]:
def plot_vancomycin_pkpd(
    dose_mg=1000,
    tau_h=12,
    infusion_h=1,
    vd_l=70,
    cl_l_h=4,
    mic_mg_l=1,
    n_doses=8
):
    if infusion_h >= tau_h:
        print("Infusion time must be shorter than dosing interval.")
        return

    t, concentration, dose_times = simulate_multiple_iv_infusions(
        dose_mg=dose_mg,
        tau_h=tau_h,
        infusion_h=infusion_h,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        n_doses=n_doses,
        dt=0.01
    )

    metrics = calculate_vancomycin_metrics(
        t=t,
        concentration=concentration,
        dose_mg=dose_mg,
        tau_h=tau_h,
        infusion_h=infusion_h,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        mic_mg_l=mic_mg_l,
        n_doses=n_doses
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(t, concentration, linewidth=2, label="Vancomycin concentration")
    ax.axhline(mic_mg_l, linestyle="--", label=f"MIC = {mic_mg_l:.1f} mg/L")

    for dose_time in dose_times:
        ax.axvline(dose_time, alpha=0.15)

    ax.set_title("Vancomycin Multiple IV Infusion Simulation")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    display(make_metrics_table(metrics))


interact(
    plot_vancomycin_pkpd,
    dose_mg=FloatSlider(value=1000, min=250, max=2500, step=250, description="Dose"),
    tau_h=Dropdown(options=[6, 8, 12, 24, 36, 48], value=12, description="Tau"),
    infusion_h=FloatSlider(value=1, min=0.5, max=4, step=0.5, description="Tinf"),
    vd_l=FloatSlider(value=70, min=30, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=4, min=0.5, max=10, step=0.5, description="CL"),
    mic_mg_l=Dropdown(options=[0.5, 1, 1.5, 2], value=1, description="MIC"),
    n_doses=IntSlider(value=8, min=2, max=20, step=1, description="Doses")
);

## 5. Observation Task 1: Effects of Dose, Interval, CL, and MIC

Complete the following exercises.

### Task A: Standard regimen

Set:

- Dose = 1000 mg
- Tau = 12 h
- Tinf = 1 h
- Vd = 70 L
- CL = 4 L/h
- MIC = 1 mg/L
- Doses = 8

Record:

- AUC24 at steady state
- AUC24/MIC
- Cmax in the last interval
- Cmin before the next dose
- Target status

### Task B: Dose increase

Change only Dose to 1500 mg.

Observe:

- Does AUC24 increase?
- Does AUC24/MIC increase?
- Does Cmin increase?
- Does the target status suggest a risk of high exposure?

### Task C: Reduced clearance

Return Dose to 1000 mg, and change only CL to 2 L/h.

Observe:

- Does AUC24 increase substantially?
- Is half-life prolonged?
- Does Cmin increase?
- Can this simulate the risk in a patient with reduced renal function?

### Task D: Increased MIC

Return CL to 4 L/h, and change only MIC to 2 mg/L.

Observe:

- Does AUC24 change?
- Does AUC24/MIC decrease?
- If the goal is AUC24/MIC ≥ 400, what AUC24 is required?
- Does this AUC24 exceed 600 mg·h/L?

## 6. Why Is MIC Important?

For vancomycin, AUC24/MIC is influenced by two factors:

$$
AUC_{24}/MIC = \frac{AUC_{24}}{MIC}
$$

If AUC24 remains unchanged:

- When MIC = 0.5 mg/L, AUC24/MIC is higher.
- When MIC = 1 mg/L, AUC24/MIC is numerically equal to AUC24.
- When MIC = 2 mg/L, AUC24/MIC is only half of the value when MIC = 1 mg/L.

This explains why, when the pathogen MIC increases, the same dosing regimen may no longer reach the target.

For example, if the target is:

$$
AUC_{24}/MIC \geq 400
$$

then the required condition is:

$$
AUC_{24} \geq 400 \times MIC
$$

When MIC = 2 mg/L, the requirement becomes:

$$
AUC_{24} \geq 800\ mg \cdot h/L
$$

This already exceeds the commonly used exposure upper limit of 600 mg·h/L. Therefore, from an educational perspective, when MIC = 2 mg/L, it may be difficult to meet both efficacy and safety goals simply by increasing the vancomycin dose.

In [ ]:
def plot_mic_effect(
    auc24=500
):
    mic_values = np.array([0.5, 1.0, 1.5, 2.0])
    auc_mic_values = auc24 / mic_values
    required_auc_values = 400 * mic_values

    result = pd.DataFrame({
        "MIC (mg/L)": mic_values,
        "AUC24 (mg*h/L)": [auc24] * len(mic_values),
        "AUC24/MIC": auc_mic_values,
        "AUC24 required for AUC/MIC >= 400": required_auc_values,
        "Feasible within AUC <= 600?": ["Yes" if x <= 600 else "No" for x in required_auc_values]
    })

    fig, ax = plt.subplots()
    ax.bar([str(x) for x in mic_values], auc_mic_values)
    ax.axhline(400, linestyle="--", label="AUC/MIC target = 400")
    ax.set_title("Effect of MIC on AUC24/MIC")
    ax.set_xlabel("MIC (mg/L)")
    ax.set_ylabel("AUC24/MIC")
    ax.legend()
    plt.show()

    display(result)


interact(
    plot_mic_effect,
    auc24=FloatSlider(value=500, min=200, max=900, step=50, description="AUC24")
);

## 7. Observation Task 2: Same AUC24, Different MIC Values

Use the MIC simulation above to complete the following tasks.

### Task A

Set:

- AUC24 = 500 mg·h/L

Record AUC24/MIC at different MIC values:

- MIC = 0.5 mg/L
- MIC = 1 mg/L
- MIC = 1.5 mg/L
- MIC = 2 mg/L

### Task B

Change AUC24 to 600 mg·h/L.

Think about:

- When MIC = 1 mg/L, can the target AUC24/MIC ≥ 400 be reached?
- When MIC = 2 mg/L, can the target AUC24/MIC ≥ 400 be reached?
- If not, should the vancomycin dose be increased without limit? Why or why not?

## 8. Renal Function Changes and Vancomycin Exposure

Vancomycin is primarily cleared by the kidneys, so changes in renal function can significantly affect CL, half-life, AUC24, and trough concentration.

This section uses the Cockcroft--Gault equation to estimate creatinine clearance:

### Male

$$
CrCl = \frac{(140-age) \times weight}{72 \times SCr}
$$

### Female

$$
CrCl = 0.85 \times \frac{(140-age) \times weight}{72 \times SCr}
$$

where:

| Symbol | Meaning | Unit |
|---|---|---|
| age | Age | years |
| weight | Body weight | kg |
| SCr | Serum creatinine | mg/dL |
| CrCl | Creatinine clearance | mL/min |

For this educational demonstration, we use a simplified relationship to estimate vancomycin clearance:

$$
CL_{vanco} = 0.5 + 0.06 \times CrCl
$$

This equation is used only for educational simulation in this notebook. It is not a clinically validated vancomycin dosing equation.

We also assume:

$$
V_d = 0.7 \times weight
$$

In real clinical practice, vancomycin dosing should be dynamically adjusted based on therapeutic drug monitoring and patient-specific factors.

In [ ]:
def cockcroft_gault(age_years, weight_kg, scr_mg_dl, sex):
    """
    Estimate creatinine clearance using Cockcroft-Gault equation.
    """
    crcl = ((140 - age_years) * weight_kg) / (72 * scr_mg_dl)
    if sex == "Female":
        crcl *= 0.85
    return crcl


def estimate_vanco_cl_from_crcl(crcl_ml_min):
    """
    Teaching-only vancomycin clearance model.
    Not intended for clinical dosing.
    """
    cl_l_h = 0.5 + 0.06 * crcl_ml_min
    return max(cl_l_h, 0.3)


def plot_renal_function_vanco(
    sex="Male",
    age_years=65,
    weight_kg=70,
    scr_mg_dl=1.0,
    dose_mg=1000,
    tau_h=12,
    infusion_h=1,
    mic_mg_l=1,
    n_doses=8
):
    if infusion_h >= tau_h:
        print("Infusion time must be shorter than dosing interval.")
        return

    crcl = cockcroft_gault(
        age_years=age_years,
        weight_kg=weight_kg,
        scr_mg_dl=scr_mg_dl,
        sex=sex
    )

    vd_l = 0.7 * weight_kg
    cl_l_h = estimate_vanco_cl_from_crcl(crcl)

    t, concentration, dose_times = simulate_multiple_iv_infusions(
        dose_mg=dose_mg,
        tau_h=tau_h,
        infusion_h=infusion_h,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        n_doses=n_doses,
        dt=0.01
    )

    metrics = calculate_vancomycin_metrics(
        t=t,
        concentration=concentration,
        dose_mg=dose_mg,
        tau_h=tau_h,
        infusion_h=infusion_h,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        mic_mg_l=mic_mg_l,
        n_doses=n_doses
    )

    target_auc = 500
    suggested_daily_dose = target_auc * cl_l_h
    suggested_dose_per_interval = suggested_daily_dose / (24 / tau_h)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(t, concentration, linewidth=2, label="Predicted concentration")
    ax.axhline(mic_mg_l, linestyle="--", label=f"MIC = {mic_mg_l:.1f} mg/L")

    for dose_time in dose_times:
        ax.axvline(dose_time, alpha=0.15)

    ax.set_title("Vancomycin Exposure and Renal Function")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    patient_table = pd.DataFrame({
        "Parameter": [
            "Sex",
            "Age",
            "Weight",
            "SCr",
            "Estimated CrCl",
            "Teaching Vd estimate",
            "Teaching CL estimate",
            "Current dose",
            "Current interval",
            "Suggested dose per interval for AUC24 ≈ 500"
        ],
        "Value": [
            sex,
            f"{age_years:.0f} years",
            f"{weight_kg:.1f} kg",
            f"{scr_mg_dl:.2f} mg/dL",
            f"{crcl:.1f} mL/min",
            f"{vd_l:.1f} L",
            f"{cl_l_h:.2f} L/h",
            f"{dose_mg:.0f} mg",
            f"q{tau_h:.0f}h",
            f"{suggested_dose_per_interval:.0f} mg q{tau_h:.0f}h"
        ]
    })

    display(patient_table)
    display(make_metrics_table(metrics))


interact(
    plot_renal_function_vanco,
    sex=Dropdown(options=["Male", "Female"], value="Male", description="Sex"),
    age_years=FloatSlider(value=65, min=18, max=95, step=1, description="Age"),
    weight_kg=FloatSlider(value=70, min=40, max=140, step=5, description="Weight"),
    scr_mg_dl=FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description="SCr"),
    dose_mg=FloatSlider(value=1000, min=250, max=2500, step=250, description="Dose"),
    tau_h=Dropdown(options=[6, 8, 12, 24, 36, 48], value=12, description="Tau"),
    infusion_h=FloatSlider(value=1, min=0.5, max=4, step=0.5, description="Tinf"),
    mic_mg_l=Dropdown(options=[0.5, 1, 1.5, 2], value=1, description="MIC"),
    n_doses=IntSlider(value=8, min=2, max=20, step=1, description="Doses")
);

## 9. Observation Task 3: Effect of Reduced Renal Function on Exposure

Use the renal function simulation above to complete the following tasks.

### Task A: Patient with relatively normal renal function

Set:

- Sex = Male
- Age = 40 years
- Weight = 70 kg
- SCr = 0.9 mg/dL
- Dose = 1000 mg
- Tau = 12 h
- MIC = 1 mg/L

Record:

- Estimated CrCl
- Teaching CL estimate
- AUC24
- Cmin before the next dose
- Target status

### Task B: Patient with reduced renal function

Change only the following parameters:

- Age = 80 years
- SCr = 2.0 mg/dL

Observe:

- Does Estimated CrCl decrease?
- Does the teaching CL estimate decrease?
- Does AUC24 increase?
- Does Cmin increase?
- Does the result suggest a need for dose adjustment or interval extension?

### Task C: Thinking about dose adjustment

Look at the table entry:

- Suggested dose per interval for AUC24 ≈ 500

Think about:

- Why does the suggested dose decrease when renal function declines?
- Why can this suggestion not directly replace real TDM?
- What additional information is needed in real clinical practice?

## 10. Regimen Comparison: Same Daily Dose, Different Dosing Intervals

For linear PK, if the total daily dose is the same and CL is unchanged, AUC24 is usually similar.

However, different dosing intervals change concentration fluctuation:

- Shorter dosing interval: smaller peak-trough fluctuation.
- Longer dosing interval: larger peak-trough fluctuation.
- Same total daily dose: AUC24 may be similar, but Cmax and Cmin may differ.

This helps illustrate that:

> AUC, Cmax, and Cmin represent different types of clinical information.

AUC reflects total exposure more directly, while Cmin is more easily influenced by dosing interval and clearance.

In [ ]:
def compare_regimens_same_daily_dose(
    daily_dose_mg=2000,
    infusion_h=1,
    vd_l=70,
    cl_l_h=4,
    mic_mg_l=1,
    n_days=4
):
    regimens = [
        {"name": "q8h", "tau_h": 8, "dose_mg": daily_dose_mg / 3},
        {"name": "q12h", "tau_h": 12, "dose_mg": daily_dose_mg / 2},
        {"name": "q24h", "tau_h": 24, "dose_mg": daily_dose_mg}
    ]

    fig, ax = plt.subplots(figsize=(10, 5))
    rows = []

    for regimen in regimens:
        tau_h = regimen["tau_h"]
        dose_mg = regimen["dose_mg"]
        n_doses = int((24 * n_days) / tau_h)

        t, concentration, dose_times = simulate_multiple_iv_infusions(
            dose_mg=dose_mg,
            tau_h=tau_h,
            infusion_h=infusion_h,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            n_doses=n_doses,
            dt=0.01
        )

        metrics = calculate_vancomycin_metrics(
            t=t,
            concentration=concentration,
            dose_mg=dose_mg,
            tau_h=tau_h,
            infusion_h=infusion_h,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            mic_mg_l=mic_mg_l,
            n_doses=n_doses
        )

        ax.plot(t, concentration, linewidth=2, label=f"{dose_mg:.0f} mg {regimen['name']}")

        rows.append({
            "Regimen": f"{dose_mg:.0f} mg {regimen['name']}",
            "Daily dose": f"{daily_dose_mg:.0f} mg/day",
            "AUC24": f"{metrics['AUC24 at steady state']:.1f} mg*h/L",
            "AUC24/MIC": f"{metrics['AUC24/MIC']:.1f}",
            "Cmax last interval": f"{metrics['Cmax in last interval']:.2f} mg/L",
            "Cmin before next dose": f"{metrics['Cmin before next dose']:.2f} mg/L",
            "Target status": metrics["Target status"]
        })

    ax.axhline(mic_mg_l, linestyle="--", label=f"MIC = {mic_mg_l:.1f} mg/L")
    ax.set_title("Comparison of Regimens with the Same Daily Dose")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    display(pd.DataFrame(rows))


interact(
    compare_regimens_same_daily_dose,
    daily_dose_mg=FloatSlider(value=2000, min=1000, max=4000, step=250, description="Daily dose"),
    infusion_h=FloatSlider(value=1, min=0.5, max=4, step=0.5, description="Tinf"),
    vd_l=FloatSlider(value=70, min=30, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=4, min=0.5, max=10, step=0.5, description="CL"),
    mic_mg_l=Dropdown(options=[0.5, 1, 1.5, 2], value=1, description="MIC"),
    n_days=IntSlider(value=4, min=2, max=7, step=1, description="Days")
);

## 11. Observation Task 4: Same Daily Dose, Different Dosing Intervals

Use the regimen comparison simulation above to complete the following tasks.

### Task A

Set:

- Daily dose = 2000 mg/day
- Tinf = 1 h
- Vd = 70 L
- CL = 4 L/h
- MIC = 1 mg/L

Compare:

- q8h
- q12h
- q24h

Observe:

- Are the AUC24 values of the three regimens similar?
- Are the Cmax values the same?
- Are the Cmin values the same?
- Which regimen has the largest peak-trough fluctuation?

### Task B

Reduce CL to 2 L/h.

Observe:

- Does AUC24 increase?
- Does Cmin increase substantially?
- When clearance decreases, why might the dosing interval need to be extended?

## 12. From AUC24 to AUC/MIC: Integrated Assessment

Vancomycin PK/PD assessment should not rely on a single number alone.

For teaching, the following sequence can be used:

1. First, check whether AUC24 reaches the exposure associated with efficacy.
2. Then, check whether AUC24/MIC reaches the target.
3. Next, check whether AUC24 exceeds the safety upper limit.
4. At the same time, observe whether Cmin is clearly elevated.
5. Combine renal function, infection severity, and TDM results to decide whether adjustment is needed.

The simplified process can be understood as follows:

$$
AUC_{24}/MIC < 400 \Rightarrow Potentially insufficient exposure
$$

$$
400 \leq AUC_{24} \leq 600\quad (MIC=1) \Rightarrow Educational target range
$$

$$
AUC_{24} > 600 \Rightarrow High exposure; nephrotoxicity risk may increase
$$

However, note that:

> In real clinical practice, vancomycin AUC usually needs to be estimated using therapeutic drug monitoring and pharmacokinetic methods, not only the simplified model in this notebook.

## 13. Self-Assessment Questions: Vancomycin PK/PD

Complete the following self-assessment questions based on this notebook. Try to answer them independently before checking the reference answers in the next cell.

---

### Question 1: What is the commonly used PK/PD index for vancomycin in serious MRSA infections?

A. Cmax/MIC  
B. AUC24/MIC  
C. %fT > MIC  
D. Tmax/MIC  

---

### Question 2: If MIC increases from 1 mg/L to 2 mg/L while AUC24 remains unchanged, how does AUC24/MIC change?

A. It increases 2-fold.  
B. It remains unchanged.  
C. It decreases to half of the original value.  
D. It decreases to one-quarter of the original value.  

---

### Question 3: For linear PK, if the total daily dose and CL are unchanged, how do the AUC24 values of q8h, q12h, and q24h regimens usually compare?

A. The AUC24 of q8h is always the largest.  
B. The AUC24 of q24h is always the largest.  
C. AUC24 is usually similar, but peak-trough fluctuation differs.  
D. Cmax and Cmin are exactly the same for all three regimens.  

---

### Question 4: When reduced renal function lowers vancomycin CL, which of the following is most likely to occur?

A. AUC24 decreases and half-life becomes shorter.  
B. AUC24 increases and half-life is prolonged.  
C. MIC automatically decreases.  
D. The drug absorption rate constant ka increases.  

---

### Question 5: If MIC = 2 mg/L, what is the minimum theoretical AUC24 needed to reach AUC24/MIC ≥ 400?

A. 200 mg·h/L  
B. 400 mg·h/L  
C. 600 mg·h/L  
D. 800 mg·h/L

## 14. Reference Answers to the Self-Assessment Questions

### Question 1

**Reference answer: B**

**Explanation:**  
For serious MRSA infections treated with vancomycin, the commonly used PK/PD index is AUC24/MIC. AUC24 reflects total exposure over 24 hours, and MIC reflects pathogen susceptibility.

---

### Question 2

**Reference answer: C**

**Explanation:**  

$$
AUC_{24}/MIC = \frac{AUC_{24}}{MIC}
$$

When AUC24 remains unchanged and MIC increases from 1 to 2, the denominator doubles, so AUC24/MIC decreases to half of the original value.

---

### Question 3

**Reference answer: C**

**Explanation:**  
In linear PK, steady-state AUC24 is mainly determined by the total daily dose and CL:

$$
AUC_{24} = \frac{Daily\ Dose}{CL}
$$

If the total daily dose and CL are the same, AUC24 is usually similar. However, different dosing intervals change Cmax, Cmin, and peak-trough fluctuation.

---

### Question 4

**Reference answer: B**

**Explanation:**  
When renal function declines, vancomycin clearance may decrease. Lower CL slows elimination, prolongs half-life, and increases AUC24 and trough concentration.

---

### Question 5

**Reference answer: D**

**Explanation:**  
If the target is:

$$
AUC_{24}/MIC \geq 400
$$

When MIC = 2 mg/L:

$$
AUC_{24} \geq 400 \times 2 = 800\ mg \cdot h/L
$$

This exceeds the commonly used exposure upper limit of 600 mg·h/L. Therefore, when MIC = 2 mg/L, it may be difficult to balance efficacy and safety simply by increasing the vancomycin dose.

## 15. Notebook Summary

This notebook used vancomycin as an example to introduce an educational simulation of antibacterial PK/PD based on AUC24/MIC.

Key takeaways:

1. AUC24/MIC is a key PK/PD index for vancomycin, especially in serious MRSA infections.
2. AUC24 reflects total drug exposure over 24 hours, while MIC reflects pathogen susceptibility.
3. Trough concentration is related to exposure, but it cannot fully replace AUC-based evaluation.
4. Dose, dosing interval, infusion duration, clearance, volume of distribution, renal function, and MIC jointly determine vancomycin exposure.
5. Higher MIC values make it more difficult to achieve PK/PD targets without exceeding the safe exposure range.
6. Simplified PK/PD simulations help explain dosing logic, but they cannot replace clinical TDM or individualized dose adjustment.

The complete logic of this section can be summarized as:

$$
Dose\ regimen \rightarrow Concentration(t) \rightarrow AUC_{24} \rightarrow AUC_{24}/MIC \rightarrow Efficacy/Safety
$$

The next section will further examine:

> Simulation of %fT > MIC for beta-lactam antibiotics.

## 16. References

1. Rybak MJ, Le J, Lodise TP, et al. Therapeutic monitoring of vancomycin for serious methicillin-resistant *Staphylococcus aureus* infections: A revised consensus guideline and review by ASHP, IDSA, PIDS, and SIDP. 2020.

2. DailyMed. Vancomycin Hydrochloride for Injection, USP. U.S. National Library of Medicine.

3. Cockcroft DW, Gault MH. Prediction of creatinine clearance from serum creatinine. *Nephron*. 1976.

Note: The equations and code in this notebook are intended for educational demonstration. The vancomycin clearance estimation component uses a simplified model and should not be used for real prescribing or TDM decisions.